# Lab 1: 実機ハードウェア向けの量子回路を構築する

## Qiskit Global Summer School 2026

Lab 1 へようこそ！このノートブックでは、量子回路をゼロから構築し、実際の量子ハードウェア上で実行できるように準備する方法を学びます。

**学ぶこと:**

1. [**パート1: 基本ゲートと量子の概念**](#part1): X、H、CX ゲートがどのように重ね合わせと量子もつれを生み出すか
2. [**パート2: 回路の深さ**](#part2): なぜ深さが重要なのか、GHZ 状態、そして対称性を使って深さを減らす方法
3. [**パート3: トランスパイル**](#part3): 実機プロセッサー（IBM の133量子ビット Heron チップ）の制約に回路を適合させる

各パートは前のパートの上に積み重なっており、最後には実機向けに効率的な64量子ビットのもつれ状態を構築できるようになります。これらのスキルは Lab 2 に直接つながります。

## 始める前に

> **このラボは全体が qBraid 上のローカルの *シミュレーター*（`FakeTorino`）で動作します。** すべての回路は C++ で構築され、ローカルでトランスパイルされ、実行されます。IBM Quantum のランタイム時間（分）は **一切** 消費しません。ただし、grader に解答を *提出* するには設定済みの IBM Quantum アカウントが必要です。採点セルで認証エラーが出た場合は、セットアップの節にあるアカウント設定セルを参照してください。

<details>
<summary><b>よくあるセットアップの問題（クリックで展開）</b></summary>

- **セットアップのセルを順番に実行してください。** パスのセルが `LAB_ROOT`/`QISKIT_CPP_SRC` を定義し、CMake のセルがビルドを構成します。カーネルを再起動したら、セットアップの節を再実行してください。
- **ビルド時に `nlohmann/json.hpp` が見つからない？** CMake の構成ステップがネットワーク経由でそれを取得します。構成時に qBraid セッションにネットワークがなかった場合は、接続が回復してから CMake のセルをもう一度実行してください。
- **採点セルが認証エラーで失敗する？** qBraid セッションは Lab 0 で保存した IBM Quantum アカウントを引き継ぎません。セットアップの節にあるアカウント設定セルを一度実行してください。
- **ASCII の回路図がずれて見える？** セットアップの節にある等幅 CSS のセルを再実行してください。
</details>

## セットアップ

パート1を始める前に、この **セットアップ** の節のセルを上から下へ一度実行してください。このラボは **qBraid** でサポートされており、C++ のラボファイルとビルド済みの `qiskit-cpp` があらかじめ環境に用意されています。

**注:** このラボは `%%writefile` セルで C++ コードを書き出します。qBraid のエディターは Python を期待しているためこれらのセルに警告を出すことがありますが、その警告は見た目だけのもので無視して構いません。

In [ ]:
# qBraid のセットアップ: qiskit-cpp のソースを見つける（または取得する）とともにラボのパスを設定する。
# このセルを最初に実行してください。後のセルは LAB_ROOT と QISKIT_CPP_SRC を参照します。
# サポートされているプラットフォームは qBraid のみです。ラボのツリーはこのノートブックに同梱されています。
# qiskit-cpp がまだ存在しない場合は、ここで（バージョン固定して）取得します。
import os, subprocess
from pathlib import Path

# このラボが検証対象としている qiskit-cpp のコミット。新しいコミットは、インストール済みの
# qiskit に存在しない Qiskit C API のシンボル（例: qk_circuit_global_phase）を呼び出すことが
# あるため、既知の良好なバージョンに固定します。
QISKIT_CPP_PIN = "91f4b5d64f69903224058521421b8305ae9d64e3"

# LAB_ROOT はこのノートブックを含むディレクトリー（C++ ラボのツリー）です。
LAB_ROOT = str(Path.cwd().resolve())

# QISKIT_CPP_SRC は qiskit-cpp の "src" ディレクトリー（ヘッダーのみの SDK）を指す必要があります。
# 解決順: 明示的な QISKIT_CPP_SRC 環境変数 -> 既知の場所 -> 上位ディレクトリーを探索。
def _find_qiskit_cpp_src():
    env = os.environ.get("QISKIT_CPP_SRC")
    if env and (Path(env) / "circuit" / "quantumcircuit.hpp").is_file():
        return str(Path(env).resolve())
    p = Path(LAB_ROOT).resolve()
    candidates = [
        Path.home() / "qiskit-cpp" / "src",
        Path("/opt/qiskit-cpp/src"),
        *[parent / "qiskit-cpp" / "src" for parent in [p, *p.parents]],
    ]
    for c in candidates:
        if (c / "circuit" / "quantumcircuit.hpp").is_file():
            return str(c.resolve())
    return None

QISKIT_CPP_SRC = _find_qiskit_cpp_src()
if QISKIT_CPP_SRC is None:
    # 存在しない（qBraid は事前読み込みしません）-- 固定バージョンを ~/qiskit-cpp にクローンします。
    dest = Path.home() / "qiskit-cpp"
    print(f"qiskit-cpp が見つかりません。固定版 {QISKIT_CPP_PIN[:8]} を {dest} にクローンします ...")
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/Qiskit/qiskit-cpp.git", str(dest)], check=True)
    subprocess.run(["git", "-C", str(dest), "checkout", "--quiet", QISKIT_CPP_PIN], check=True)
    QISKIT_CPP_SRC = _find_qiskit_cpp_src()

print(f"LAB_ROOT       : {LAB_ROOT}")
if QISKIT_CPP_SRC:
    print(f"QISKIT_CPP_SRC : {QISKIT_CPP_SRC}")
else:
    raise RuntimeError(
        "qiskit-cpp を見つけることも取得することもできませんでした。明示的に設定して再実行してください。例:\n"
        "    import os; os.environ['QISKIT_CPP_SRC'] = '/path/to/qiskit-cpp/src'"
    )

# 後で C++ ビルドが未知の qk_* シンボルで失敗する場合、既存の qiskit-cpp のチェックアウトが
# このラボの対象より新しいということです -- 固定してください:
#     git -C ~/qiskit-cpp checkout 91f4b5d6

In [ ]:
# ストリーム出力に厳密な等幅 + 横スクロールを強制して、C++ バイナリの ASCII 回路図が
# qBraid の JupyterLab で列がそろって表示されるようにします。
from IPython.display import HTML, display
display(HTML("""
<style>
.jp-OutputArea-output pre,
.jupyter-widgets-output-area pre,
.output pre {
    white-space: pre !important;
    overflow-x: auto !important;
    font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace !important;
}
</style>
"""))

In [ ]:
# これらのパッケージが qBraid 環境にまだインストールされていない場合は、このセルを実行してください。
# (%pip マジックを使うことで、パッケージは *実行中の* カーネルにインストールされます。
#  素の !pip が間違った環境を対象にしうる qBraid では重要です。)
%pip install "qiskit>=2.4.1" qiskit-ibm-runtime qiskit-aer numpy networkx matplotlib qiskit-qasm3-import
%pip install --upgrade git+https://github.com/qiskit-community/Quantum-Challenge-Grader.git

In [ ]:
# qBraid では C++ のラボファイルはすでにこのディレクトリー（LAB_ROOT）に存在します。
# ビルドを構成する前に、ラボのツリーが正しく見えることを確認します。
from pathlib import Path
if (Path(LAB_ROOT) / "CMakeLists.txt").is_file():
    print(f"ラボファイルが {LAB_ROOT} に見つかりました")
else:
    raise RuntimeError(
        f"{LAB_ROOT} に C++ ラボのツリー（CMakeLists.txt）が見つかりませんでした。"
        "qBraid 上のラボディレクトリーの中からこのノートブックを開いてください。"
    )

In [ ]:
# qiskit-cpp は上のパスのセルで見つけました（QISKIT_CPP_SRC）。ここで確認します。
from pathlib import Path
assert QISKIT_CPP_SRC and (Path(QISKIT_CPP_SRC) / "circuit" / "quantumcircuit.hpp").is_file(), (
    f"qiskit-cpp のヘッダーが QISKIT_CPP_SRC={QISKIT_CPP_SRC!r} に見つかりません。"
    "上のパスのセルを再実行してください。"
)
print(f"qiskit-cpp のヘッダー : {QISKIT_CPP_SRC}")

In [ ]:
# CMake がすべてのターゲットを構成できるように、例ごとのディレクトリーとプレースホルダーの
# ソースファイルが存在することを保証します。非破壊的: 既存のソースファイルは上書きしません。
EXAMPLES = [
    'x_gate_circuit', 'h_gate_circuit', 'bell_state', 'bell_exercise',
    'bell_phase', 'bell_phase_exercise', 'ghz_3qubit', 'ghz_fan_chain',
    'depth_examples', 'depth_exercise', 'ghz_depth', 'ghz_half_depth',
    'ghz_exercise', 'bridge_verify', 'bridge_exercise', 'ghz_line_exercise',
    'ghz_hex_exercise', 'ghz_64_exercise', 'transpile_setup', 'transpile_demo',
    'topology_viz',
]

for name in EXAMPLES:
    d = Path(LAB_ROOT) / name
    d.mkdir(parents=True, exist_ok=True)
    src = d / f'{name}.cpp'
    if not src.exists():
        src.touch()

print(f'ラボのディレクトリーが {LAB_ROOT} の下に準備できました')

In [ ]:
# ビルドを構成します。CMake の `python3` 探索が Qiskit（その C API のヘッダー/ライブラリ）を持つ
# インタープリターに解決されるよう、実行中カーネルの Python を PATH の先頭に置きます。
# スペースを含むディレクトリーでも動くよう、パスは引用符で囲みます。
import os, sys
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
!cmake -S "{LAB_ROOT}" -B "{LAB_ROOT}/build" -DQISKIT_CPP_SRC="{QISKIT_CPP_SRC}"

In [ ]:
# grader クライアントと OpenQASM 3 リーダーをインポートします（採点セルで使用します）。
from qc_grader.challenges.qgss_2026 import (
    grade_lab1_ex1,
    grade_lab1_ex2,
    grade_lab1_ex3,
    grade_lab1_ex4,
    grade_lab1_ex5,
    grade_lab1_ex6,
    grade_lab1_ex7,
)

### (任意) IBM Quantum アカウントのセットアップ

grader は解答をローカルで検証しますが、それを *提出* するには設定済みの IBM Quantum アカウントが必要です。新しい qBraid セッションは Lab 0 で保存したアカウントを引き継ぎません。採点セルが認証エラーを報告する場合は、下のセルを一度実行してください。

In [ ]:
# 採点が認証エラーで失敗する場合は、qBraid 環境ごとにこれを一度だけ実行してください。
# Lab 0 で使ったのと同じ save_account(...) 呼び出しを使ってください。例えば:
#
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<YOUR_IBM_QUANTUM_API_TOKEN>",
#     overwrite=True,
# )

---

## パート1: 基本ゲートと量子の概念 <a id="part1"></a>

量子計算は、古典ビットの量子版である **量子ビット** を操作することで動作します。量子ビットは状態 $|0\rangle$ から始まり、それを **量子ゲート** を使って変換します。最も重要な単一量子ビットゲートと2量子ビットゲートの3つを見ていきましょう。

### X ゲート（ビット反転）

X ゲートは量子ビットの状態を反転させます。$|0\rangle \to |1\rangle$、$|1\rangle \to |0\rangle$ です。古典的な NOT ゲートの量子版です。

幾何学的には、X はブロッホ球の X 軸まわりの $\pi$（180°）回転です。その行列は次のとおりです。

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$

In [ ]:
%%writefile "{LAB_ROOT}/x_gate_circuit/x_gate_circuit.cpp"
// 単一量子ビットに X ゲートを適用する回路を構築する

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(1, 0);

    circ.x(0);
    circ.draw();

    auto sv = Statevector::from_instruction(circ);
    std::cout << "X|0> = " << sv << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make x_gate_circuit && ./x_gate_circuit

### H ゲート（重ね合わせ）

アダマールゲートは量子ビットを **重ね合わせ** にします。これは、同時に一部が $|0\rangle$、一部が $|1\rangle$ である状態です。

$$H|0\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr)$$

この状態の量子ビットを測定すると、$|0\rangle$ または $|1\rangle$ がそれぞれ 50% の確率で得られます。X と同様に H ゲートも回転で、今回はブロッホ球上で X 軸と Z 軸のちょうど中間にある軸まわりの $\pi$ 回転です。その行列は次のとおりです。

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

In [ ]:
%%writefile "{LAB_ROOT}/h_gate_circuit/h_gate_circuit.cpp"
// 単一量子ビットに H ゲートを適用する回路を構築する

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(1, 0);

    circ.h(0);
    circ.draw();

    auto sv = Statevector::from_instruction(circ);
    std::cout << "H|0> = " << sv << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make h_gate_circuit && ./h_gate_circuit

### CX（CNOT）ゲートと量子もつれ

CX ゲートは **2量子ビット** ゲートです。**制御** 量子ビットが $|1\rangle$ のとき、かつそのときに限り **標的** 量子ビットを反転させます。制御が重ね合わせにあるとき、注目すべきことが起こります。2つの量子ビットが **もつれる** のです。それらの測定結果は、古典的には説明できないかたちで完全に相関するようになります。

最初の **ベル状態** を作って、これを実際に見てみましょう。

In [ ]:
%%writefile "{LAB_ROOT}/bell_state/bell_state.cpp"
// ベル状態 (|00> + |11>) / sqrt(2) を作成する

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(2, 0);

    circ.h(0);       // 量子ビット0を重ね合わせにする
    circ.cx(0, 1);   // 量子ビット1を量子ビット0ともつれさせる
    circ.draw();

    auto sv = Statevector::from_instruction(circ);
    std::cout << "Bell state: " << sv << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make bell_state && ./bell_state

#### ベル状態を理解する

今作った状態は $\frac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr)$ です。これは何を意味するのでしょうか。

- 両方の量子ビットを測定すると、$|00\rangle$ または $|11\rangle$ がそれぞれ 50% の確率で得られます。
- $|01\rangle$ や $|10\rangle$ は **決して** 得られません。量子ビットは完全に相関しています。
- この相関は、量子ビットがどれだけ離れていても持続します。これが **量子もつれ** であり、量子計算を強力にする鍵となるリソースです。

### 演習1: $|01\rangle + |10\rangle$ のベル状態を構築する

先ほど $\frac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr)$ を構築しました。今度はあなたの番です！

**課題:** ベル状態 $\frac{1}{\sqrt{2}}\bigl(|01\rangle + |10\rangle\bigr)$ を準備する回路を構築してください。

**ヒント:** $|00\rangle + |11\rangle$ の作り方はもう知っています。片方の量子ビットを反転させるには、どのゲートを1つ追加すればよいでしょうか。

In [ ]:
%%writefile "{LAB_ROOT}/bell_exercise/bell_exercise.cpp"
// 演習1: |01> + |10> のベル状態を構築する

#include "compat/quantumcircuit.hpp"
#include <fstream>
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(2, 0);

    // TODO: ここにあなたのコードを追加してください

    circ.draw();

    // Python の qc_grader クライアントで採点するために OpenQASM 3 を出力します。
    std::ofstream qasm_out("../bell_exercise/circuit.qasm");
    qasm_out << circ.to_qasm3();
    qasm_out.close();

    auto sv = Statevector::from_instruction(circ);
    std::cout << "Your state: " << sv << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make bell_exercise && ./bell_exercise

In [ ]:
# C++ バイナリが書き出した QASM3 を再読み込みし、再構成した回路を既存の grader に
# 提出することで解答を採点します。
import qiskit.qasm3
from pathlib import Path
_qasm_path = Path(LAB_ROOT) / 'bell_exercise' / 'circuit.qasm'
_qasm = _qasm_path.read_text() if _qasm_path.is_file() else ''
if not _qasm.strip():
    raise RuntimeError(
        f"{_qasm_path} が空か存在しません -- 採点する前に、（コードを埋めた状態で）"
        "上のビルド/実行セルを実行してください。"
    )
qc = qiskit.qasm3.loads(_qasm)
grade_lab1_ex1(qc)

#### 同じ状態への複数の道すじ

量子回路の美しい点の一つは、同じ状態がしばしば異なる方法で準備できることです。$|01\rangle + |10\rangle$ を構築する方法は（少なくとも）3通りあります。

<details>
<summary><b>クリックして3つの解答をすべて表示</b></summary>

**方法1:** ベル回路の *前* に X を適用する:

```cpp
circ.x(1);        // 先に量子ビット1を |1> に反転
circ.h(0);
circ.cx(0, 1);
```

**方法2:** ベル回路の *後* に量子ビット1へ X を適用する:

```cpp
circ.h(0);
circ.cx(0, 1);
circ.x(1);        // もつれさせた後に量子ビット1を反転
```

**方法3:** ベル回路の *後* に量子ビット0へ X を適用する:

```cpp
circ.h(0);
circ.cx(0, 1);
circ.x(0);        // もつれさせた後に量子ビット0を反転
```

3つとも同じ状態を生成します！これは意外に思えるかもしれません。特に、量子ビット1ではなく量子ビット0を反転させる方法3はそうです。しかし量子ビットがもつれているため、*どちらか* 一方を反転させると $|00\rangle \leftrightarrow |01\rangle$ と $|11\rangle \leftrightarrow |10\rangle$ が入れ替わり、どちらの場合も $|01\rangle + |10\rangle$ が得られます。

この考え方（同じ量子状態が異なる回路で到達できること）は、実機向けの最適化を始めるときに重要になります。
</details>

### 任意: 相対位相

ここまで $|00\rangle + |11\rangle$ と $|01\rangle + |10\rangle$ を見てきました。しかし実はベル状態は **4つ** あります。残りの2つには **マイナス符号**（相対位相）が含まれます。

$$\frac{1}{\sqrt{2}}\bigl(|00\rangle - |11\rangle\bigr) \qquad \text{および} \qquad \frac{1}{\sqrt{2}}\bigl(|01\rangle - |10\rangle\bigr)$$

**Z ゲート** はこの位相を導入します。$|0\rangle$ はそのままにし、$|1\rangle \to -|1\rangle$ に写します。

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

このマイナス符号は標準基底での測定確率を変えません（依然として 50/50 になります）が、物理的には実在し、その後の計算での状態の干渉のしかたに影響します。

In [ ]:
%%writefile "{LAB_ROOT}/bell_phase/bell_phase.cpp"
// Z ゲートを追加して |00> - |11> を作成する

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(2, 0);

    circ.h(0);
    circ.cx(0, 1);
    circ.z(0);         // circ.z(1) に変えてみてください -- 同じ結果になります！
    circ.draw();

    auto sv = Statevector::from_instruction(circ);
    std::cout << "|00> - |11>: " << sv << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make bell_phase && ./bell_phase

**任意の発展演習:** $\frac{1}{\sqrt{2}}\bigl(|01\rangle - |10\rangle\bigr)$ を構築できますか？ X ゲートと Z ゲートについて学んだことを組み合わせてみてください。（解答は用意していません。試してみましょう！）

In [ ]:
%%writefile "{LAB_ROOT}/bell_phase_exercise/bell_phase_exercise.cpp"
// 任意: |01> - |10> を構築する

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(2, 0);

    // TODO: ここにあなたのコードを追加してください

    circ.draw();

    auto sv = Statevector::from_instruction(circ);
    std::cout << "Your state: " << sv << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make bell_phase_exercise && ./bell_phase_exercise

### パート1のまとめ

**達成した学習目標:**
- **X ゲート**: $|0\rangle \leftrightarrow |1\rangle$ を反転（ビット反転 / X 軸まわりの $\pi$ 回転）
- **H ゲート**: 重ね合わせ $|0\rangle \to \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ を生成
- **CX ゲート**: 2つの量子ビットをもつれさせる、量子的優位性への鍵
- 同じ量子状態は **複数の異なる回路** で準備できる
- *任意*: 相対位相（$\pm$ 符号）は物理的に意味を持つ

---

## パート2: 回路の深さ <a id="part2"></a>

実際の量子ハードウェアでは、すべてのゲートがわずかな誤差を持ち込みます。ゲートの層が多い回路ほど、より多くのノイズを蓄積し、より悪い結果を生みます。この層の数は **回路の深さ** と呼ばれ、それを減らすことは量子計算における最も重要な実践的スキルの一つです。

このパートでは、深さとは何かを学び、より大きなもつれ状態を構築し、**対称性** によって回路の深さを劇的に減らせることを発見します。

### GHZ 状態: 大規模な量子もつれ

ベル状態は2つの量子ビットをもつれさせます。**GHZ 状態**（Greenberger–Horne–Zeilinger）は、これを $N$ 量子ビットに一般化します。

$$|\text{GHZ}_N\rangle = \frac{1}{\sqrt{2}}\bigl(|00\ldots0\rangle + |11\ldots1\rangle\bigr)$$

3量子ビットの場合: $\frac{1}{\sqrt{2}}\bigl(|000\rangle + |111\rangle\bigr)$。測定すると、すべての量子ビットが一致します。すべて0か、すべて1のいずれかです。

In [ ]:
%%writefile "{LAB_ROOT}/ghz_3qubit/ghz_3qubit.cpp"
// 3量子ビットの GHZ 状態

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(3, 0);

    circ.h(0);
    circ.cx(0, 1);
    circ.cx(1, 2);
    circ.draw();

    auto sv = Statevector::from_instruction(circ);
    std::cout << "3-qubit GHZ: " << sv << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_3qubit && ./ghz_3qubit

### N 量子ビットの GHZ 状態を構築する2つの方法

GHZ 状態を構築する自然な方法は（少なくとも）2つあります。

1. **量子ビット0からのファンアウト**: 量子ビット0に H を適用し、次に量子ビット0から他のすべての量子ビットへ CX を適用します。CX(0,1)、CX(0,2)、…、CX(0,N−1)。
2. **隣接に沿ったチェーン**: 量子ビット0に H を適用し、次にもつれをチェーンに沿って伝えます。CX(0,1)、CX(1,2)、…、CX(N−2,N−1)。

どちらもまったく同じ GHZ 状態を生成します。再利用可能な関数として実装しましょう。

In [ ]:
%%writefile "{LAB_ROOT}/ghz_fan_chain/ghz_fan_chain.cpp"
// N 量子ビットの GHZ 状態を構築する2つの方法

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

QuantumCircuit ghz_fan(int n) {
    QuantumCircuit circ(n, 0);
    circ.h(0);
    for (int i = 1; i < n; i++)
        circ.cx(0, i);
    return circ;
}

QuantumCircuit ghz_chain(int n) {
    QuantumCircuit circ(n, 0);
    circ.h(0);
    for (int i = 0; i < n - 1; i++)
        circ.cx(i, i + 1);
    return circ;
}

int main() {
    std::cout << "Fan-out (5 qubits):" << std::endl;
    auto fan = ghz_fan(5);
    fan.draw();

    std::cout << "\nChain (5 qubits):" << std::endl;
    auto chain = ghz_chain(5);
    chain.draw();

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_fan_chain && ./ghz_fan_chain

### 回路の深さとは何か？

量子回路の **深さ** は、できるだけ多くのゲートを **並列に** 実行すると仮定したときに、その回路を実行するのに必要な時間ステップ（層）の数です。

**並列化のルール:**
- 2つのゲートが **同じ層** で実行できるのは、それらが **完全に異なる量子ビット** に作用する場合だけです。
- 2つのゲートが量子ビットを1つでも共有する場合、それらは **異なる層** で実行しなければなりません。

> **注意:** 回路の見た目の描画は誤解を招くことがあります！図の上ではゲートが横並びに見えても、量子ビットを共有しているために順次実行になることがあります。深さが重要なときは、常にコードで確認してください。

In [ ]:
%%writefile "{LAB_ROOT}/depth_examples/depth_examples.cpp"
// 回路の深さの例

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    // 例1: 独立した2つの X ゲート -> 深さ1（並列）
    QuantumCircuit qc1(2, 0);
    qc1.x(0);
    qc1.x(1);
    std::cout << "Two independent X gates -- depth: "
              << qc1.depth() << std::endl;
    qc1.draw();

    // 例2: 量子ビット0を共有する2つの CX ゲート -> 深さ2（順次実行）
    QuantumCircuit qc2(3, 0);
    qc2.cx(0, 1);
    qc2.cx(0, 2);
    std::cout << "\nCX(0,1) then CX(0,2) -- depth: "
              << qc2.depth() << std::endl;
    qc2.draw();

    // 例3: 互いに素な量子ビット上の2つの CX ゲート -> 深さ1（並列）
    QuantumCircuit qc3(4, 0);
    qc3.cx(0, 1);
    qc3.cx(2, 3);
    std::cout << "\nCX(0,1) and CX(2,3) in parallel -- depth: "
              << qc3.depth() << std::endl;
    qc3.draw();

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make depth_examples && ./depth_examples

### 演習2: 回路の深さを予測する

下の回路を見て、チェックセルを実行する前に **その深さを予測** してください。どのゲートが並列に実行できるかを考えましょう。

**ヒント:** 各ゲートがどの量子ビットに触れるかを書き出して、層を割り出しましょう。

In [ ]:
%%writefile "{LAB_ROOT}/depth_exercise/depth_exercise.cpp"
// 演習2: 回路の深さを予測する
// 各回路を見て、出力を読む前にその深さを予測しましょう！

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    // 回路 A
    QuantumCircuit qc_a(4, 0);
    qc_a.h(0);
    qc_a.h(1);
    qc_a.cx(0, 2);
    qc_a.cx(1, 3);
    qc_a.cx(2, 3);
    std::cout << "Circuit A:" << std::endl;
    qc_a.draw();

    // 回路 B
    QuantumCircuit qc_b(3, 0);
    qc_b.h(0);
    qc_b.cx(0, 1);
    qc_b.h(2);
    qc_b.cx(2, 1);
    std::cout << "\nCircuit B:" << std::endl;
    qc_b.draw();

    // 予測を確認しましょう！
    std::cout << "\n--- Answers ---" << std::endl;
    std::cout << "Circuit A depth: " << qc_a.depth() << std::endl;
    std::cout << "Circuit B depth: " << qc_b.depth() << std::endl;

    return 0;
}

In [ ]:
# このセルを実行する前に、あなたの予測を記入してください。
your_answer_a = 0  # あなたの予測: 回路 A の深さ
your_answer_b = 0  # あなたの予測: 回路 B の深さ

grade_lab1_ex2({"a": your_answer_a, "b": your_answer_b})

In [ ]:
!cd "{LAB_ROOT}/build" && make depth_exercise && ./depth_exercise

<details>
<summary><b>クリックして解説を表示</b></summary>

**回路 A（深さ 3）:**
- 層1: H(0) と H(1)（異なる量子ビット、並列）
- 層2: CX(0,2) と CX(1,3)（異なる量子ビット、並列）
- 層3: CX(2,3)

**回路 B（深さ 3）:**
- 層1: H(0) と H(2)（異なる量子ビット、並列）
- 層2: CX(0,1)（H(0) を待たなければならない）
- 層3: CX(2,1)（前の CX と量子ビット1を共有するため、順次実行になる）

回路 B の描画では CX(0,1) と CX(2,1) が並列にできそうに見えるかもしれませんが、どちらも量子ビット1を使っていることに注目してください！
</details>

### 私たちの GHZ 回路の深さ

先ほど構築した2つの GHZ 構成の深さを確認しましょう。一方がもう一方より優れていると予想するかもしれませんが……

In [ ]:
%%writefile "{LAB_ROOT}/ghz_depth/ghz_depth.cpp"
// ファンアウト vs チェーンの GHZ 構成の深さを比較する

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

QuantumCircuit ghz_fan(int n) {
    QuantumCircuit circ(n, 0);
    circ.h(0);
    for (int i = 1; i < n; i++)
        circ.cx(0, i);
    return circ;
}

QuantumCircuit ghz_chain(int n) {
    QuantumCircuit circ(n, 0);
    circ.h(0);
    for (int i = 0; i < n - 1; i++)
        circ.cx(i, i + 1);
    return circ;
}

int main() {
    for (int n : {5, 10, 20}) {
        auto fan = ghz_fan(n);
        auto chain = ghz_chain(n);
        std::cout << "n=" << n
                  << ":  fan depth = " << fan.depth()
                  << ",  chain depth = " << chain.depth()
                  << std::endl;
    }

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_depth && ./ghz_depth

どちらも **同じ深さ** です。両方とも $N$（H ゲートで1 + CX ゲートで $N-1$）です。なぜでしょうか。

- **ファンアウト**: すべての CX ゲートが量子ビット0を制御として共有するので、どれも並列に実行できません。
- **チェーン**: 各 CX は前の CX の結果に依存します（量子ビット $i$ がもつれてからでないと、量子ビット $i+1$ をもつれさせられません）。

どちらの構成も量子ビット数に対して *線形* です。100量子ビットなら100層のゲートで、ノイズの多いハードウェアには多すぎます。もっとうまくできるでしょうか。

### 深さを減らす: 真ん中から始める

ここで鍵となる洞察です。GHZ 状態は **対称的** です。すべての量子ビットが同じ重ね合わせに行き着きます。一方の端から構築し始める必要はないのです！

H ゲートを **真ん中** の量子ビットに置き、両方向へ同時にファンアウトすると、2つの半分が並列に進みます。これにより CX の深さがおよそ **半分** になります。

In [ ]:
%%writefile "{LAB_ROOT}/ghz_half_depth/ghz_half_depth.cpp"
// 真ん中から始め、両方向へファンアウトする GHZ

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

QuantumCircuit ghz_fan(int n) {
    QuantumCircuit circ(n, 0);
    circ.h(0);
    for (int i = 1; i < n; i++)
        circ.cx(0, i);
    return circ;
}

QuantumCircuit ghz_chain(int n) {
    QuantumCircuit circ(n, 0);
    circ.h(0);
    for (int i = 0; i < n - 1; i++)
        circ.cx(i, i + 1);
    return circ;
}

QuantumCircuit ghz_half(int n) {
    QuantumCircuit circ(n, 0);
    int mid = n / 2;
    circ.h(mid);

    for (int step = 1; step < n; step++) {
        int left = mid - step;
        int right = mid + step;
        if (left >= 0)
            circ.cx(left + 1, left);
        if (right < n)
            circ.cx(right - 1, right);
        if (left <= 0 && right >= n - 1)
            break;
    }
    return circ;
}

int main() {
    auto qc_half = ghz_half(20);
    qc_half.draw();

    std::cout << "Half-depth GHZ (20 qubits): depth "
              << qc_half.depth() << std::endl;
    auto qc_chain = ghz_chain(20);
    std::cout << "Linear GHZ (20 qubits):     depth "
              << qc_chain.depth() << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_half_depth && ./ghz_half_depth

### さらに進める: 再帰的ファンアウト

半分にするトリックは、この考え方を一度だけ適用したものです。作業を2つの枝に分割しました。しかしこれを **再帰的に** 適用できます。真ん中の量子ビットがその左右の量子ビットをもつれさせたら、新しくもつれたそれらの量子ビットが *それぞれ* 新しい起点となって、もつれをさらに広げられるのです。

各層で、もつれていない隣接量子ビットを持つ、すでにもつれた各量子ビットが1つの CX を実行します。これにより、層ごとにもつれた量子ビットの数が **2倍** になります。

| 層 | 動作 | もつれた量子ビット |
|-------|--------|-----------------|
| 0 | 量子ビット0に H | 1 |
| 1 | CX(0,1) | 2 |
| 2 | CX(0,2) + CX(1,3) | 4 |
| 3 | CX(0,4) + CX(1,5) + CX(2,6) + CX(3,7) | 8 |
| 4 | CX(0,8) + CX(1,9) + … + CX(7,15) | 16 |

深さは $N$ ではなく $\lceil\log_2(N)\rceil + 1$ として増加します。**指数的な改善** です！

$N = 16$ の場合: 深さ = $\log_2(16) + 1 = 5$。

### 演習3: 16量子ビットの深さ5の GHZ 状態を構築する

**課題:** 再帰的ファンアウトを使って、深さがちょうど5の16量子ビット GHZ 回路を構築してください。

**戦略:** 各 CX 層で、すでにもつれた各量子ビットが新しい量子ビットを1つもつれさせるようにします。$k$ 回の CX 層の後には、$2^k$ 個のもつれた量子ビットになっているはずです。

**注:** この抽象的な（ハードウェアによらない）設定では、どの量子ビットも他のどの量子ビットにも接続できます。ハードウェアの制約はパート3で扱います。

In [ ]:
%%writefile "{LAB_ROOT}/ghz_exercise/ghz_exercise.cpp"
// 演習3: 16量子ビットの深さ5の GHZ 状態を構築する

#include "compat/quantumcircuit.hpp"
#include <fstream>
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(16, 0);

    // 層0: アダマール -- 1量子ビットを重ね合わせに
    circ.h(0);

    // 層1: 1個の CX ゲート -> 2個のもつれた量子ビット
    // TODO: ここにあなたのコードを追加してください

    // 層2: 2個の CX ゲート -> 4個のもつれた量子ビット
    // TODO: ここにあなたのコードを追加してください

    // 層3: 4個の CX ゲート -> 8個のもつれた量子ビット
    // TODO: ここにあなたのコードを追加してください

    // 層4: 8個の CX ゲート -> 16個のもつれた量子ビット
    // TODO: ここにあなたのコードを追加してください

    circ.draw();

    auto d = circ.depth();
    std::cout << "Depth: " << d << std::endl;

    // 確認: 有効な GHZ 状態かどうかをチェック
    // Python の qc_grader クライアントで採点するために OpenQASM 3 を出力します。
    std::ofstream qasm_out("../ghz_exercise/circuit.qasm");
    qasm_out << circ.to_qasm3();
    qasm_out.close();

    auto sv = Statevector::from_instruction(circ);
    auto data = sv.data();
    double prob_0 = std::norm(data[0]);
    double prob_last = std::norm(data.back());

    if (d == 5 && std::abs(prob_0 - 0.5) < 0.01 && std::abs(prob_last - 0.5) < 0.01) {
        std::cout << "Valid 16-qubit GHZ state with depth 5!" << std::endl;
    } else {
        std::cout << "Not quite right. Check your CX gates." << std::endl;
    }

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_exercise && ./ghz_exercise

In [ ]:
# C++ バイナリが書き出した QASM3 を再読み込みし、再構成した回路を既存の grader に
# 提出することで解答を採点します。
import qiskit.qasm3
from pathlib import Path
_qasm_path = Path(LAB_ROOT) / 'ghz_exercise' / 'circuit.qasm'
_qasm = _qasm_path.read_text() if _qasm_path.is_file() else ''
if not _qasm.strip():
    raise RuntimeError(
        f"{_qasm_path} が空か存在しません -- 採点する前に、（コードを埋めた状態で）"
        "上のビルド/実行セルを実行してください。"
    )
qc = qiskit.qasm3.loads(_qasm)
grade_lab1_ex3(qc)

### パート2のまとめ

**達成した学習目標:**
- **回路の深さ**: 順次実行されるゲート層の数（少ないほど = ノイズが少ない）
- **GHZ 状態**: $N$ 量子ビットの量子もつれ $\frac{1}{\sqrt{2}}(|00\ldots0\rangle + |11\ldots1\rangle)$
- 線形の GHZ 回路（ファンアウトまたはチェーン）は深さ $N$
- **真ん中から始める** と深さが半分になる
- **再帰的ファンアウト** は深さ $\lceil\log_2(N)\rceil + 1$ を達成し、指数的に優れている

---

## パート3: トランスパイル <a id="part3"></a>

ここまで、抽象的な量子ビットと H や CX といったなじみのあるゲートを使って回路を構築してきました。しかし実際の量子ハードウェアには具体的な制約があります。

1. **限られたネイティブゲート**: プロセッサーは固定されたゲートの集合しかサポートしません。IBM Heron プロセッサーは $\{R_Z, \sqrt{X}, X, CZ\}$ を使います。H ゲートも CX ゲートもありません！
2. **限られた接続性**: すべての量子ビットのペアが物理的に接続されているわけではありません。接続されていない量子ビット間で2量子ビットゲートが必要な場合、追加の SWAP ゲートを挿入しなければなりません（それぞれ 3 個の CZ ゲートのコストがかかります）。
3. **物理量子ビットの割り当て**: あなたの抽象的な量子ビット0は、チップ上の特定の物理量子ビットに対応づけられなければなりません。

**トランスパイル** は、抽象的な回路をこれらすべての制約を尊重する回路へと変換する処理です。実際に見てみましょう。

In [ ]:
%%writefile "{LAB_ROOT}/transpile_setup/transpile_setup.cpp"
// FakeTorino 相当のバックエンド情報

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"
#include "compat/target.hpp"
#include "compat/transpile.hpp"

using namespace Qiskit::compat;

int main() {
    auto target = make_torino_target();

    std::cout << "Backend: FakeTorino (Heron processor)" << std::endl;
    std::cout << "Qubits: " << target.num_qubits() << std::endl;

    auto gates = torino_basis_gates();
    std::cout << "Native gates:";
    for (auto& g : gates) std::cout << " " << g;
    std::cout << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make transpile_setup && ./transpile_setup

In [ ]:
%%writefile "{LAB_ROOT}/transpile_demo/transpile_demo.cpp"
// シンプルな5量子ビットのチェーン GHZ 回路をトランスパイルする

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"
#include "compat/target.hpp"
#include "compat/transpile.hpp"

using namespace Qiskit::compat;

QuantumCircuit ghz_chain(int n) {
    QuantumCircuit circ(n, 0);
    circ.h(0);
    for (int i = 0; i < n - 1; i++)
        circ.cx(i, i + 1);
    return circ;
}

int main() {
    auto target = make_torino_target();
    auto qc = ghz_chain(5);

    std::cout << "Original circuit:" << std::endl;
    qc.draw();

    auto qc_meas = qc.measure_all();

    auto pm = generate_preset_pass_manager(1, target);
    auto isa_qc = pm.run(qc_meas);

    std::cout << "\nTranspiled circuit:" << std::endl;
    isa_qc.draw(false);

    auto ops_before = qc.count_ops();
    auto ops_after = isa_qc.count_ops();

    std::cout << "\nBefore: depth " << qc.depth() << ", gates {";
    for (auto& kv : ops_before)
        std::cout << " " << kv.first << ":" << kv.second;
    std::cout << " }" << std::endl;

    std::cout << "After:  depth " << isa_qc.depth() << ", gates {";
    for (auto& kv : ops_after)
        std::cout << " " << kv.first << ":" << kv.second;
    std::cout << " }" << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make transpile_demo && ./transpile_demo

#### 何が変わったのか？

トランスパイル後の回路を見てください。

- **H が消え**、$R_Z$ と $\sqrt{X}$ ゲート（ネイティブな単一量子ビットゲート）の組み合わせに置き換わっています
- **CX が消え**、単一量子ビットゲートに囲まれた CZ ゲートに置き換わっています
- **物理量子ビットの番号** が現れます。あなたの抽象的な量子ビットは、今や特定のハードウェア量子ビットに対応づけられています
- **深さが増加しました**: 各抽象ゲートが複数のネイティブゲートに展開されます

良い知らせ: トランスパイラーは5つの量子ビットすべてがチェーン状に物理的に接続されるレイアウトを見つけたので、**SWAP ゲートは不要でした**。いつもそうとは限りません！

### ブリッジゲートの等式

SWAP は非局所的な CX を扱う唯一の方法ではありません。**ブリッジゲート** と呼ばれる巧妙な回路の等式があり、チェーン `0 - 1 - 2` 上で最近接ゲートだけを使って、**状態を一切移動させずに** `CX(0, 2)` を実装します。

$$\text{CX}(0, 2) \;=\; \text{CX}(0, 1)\,\cdot\,\text{CX}(1, 2)\,\cdot\,\text{CX}(0, 1)\,\cdot\,\text{CX}(1, 2)$$

**時間順に左から右へ読みます:** まず `CX(0,1)`、次に `CX(1,2)`、次にもう一度 `CX(0,1)`、そして再び `CX(1,2)` を適用します。`·` は単に「その次に」を意味します。

この順序での4つの最近接 CX ゲートは、1つの遠距離 `CX(0, 2)` と **まったく同じ** 変換を生成します。真ん中の量子ビットは「ブリッジ（橋）」として働きます。その状態は一時的に反転され、その後元に戻されて変化しないまま、もつれの作用が量子ビット0から量子ビット2へ伝播します。

ブリッジは、*1つ* の中間量子ビットを通して *1つ* の非局所 CX が必要なときに役立ちます。より複雑なルーティングでは、トランスパイラーは通常やはり SWAP を好みます。しかしこの等式は、非局所ゲートのコストを具体的に示します。このトリックを使っても、「無料の」非局所 CX は、局所的なものに比べて **4倍** の数の CX ゲートがかかります。

In [ ]:
%%writefile "{LAB_ROOT}/bridge_verify/bridge_verify.cpp"
// ブリッジの等式を検証: CX(0, 2) == CX(0,1) CX(1,2) CX(0,1) CX(1,2)

#include "compat/quantumcircuit.hpp"
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    // 直接: 1つの遠距離 CX（もつれの作用が見えるように量子ビット0に H を付ける）
    QuantumCircuit qc_direct(3, 0);
    qc_direct.h(0);
    qc_direct.cx(0, 2);

    // ブリッジ: 4つの最近接 CX ゲート
    QuantumCircuit qc_bridge(3, 0);
    qc_bridge.h(0);
    qc_bridge.cx(0, 1);
    qc_bridge.cx(1, 2);
    qc_bridge.cx(0, 1);
    qc_bridge.cx(1, 2);

    auto sv_direct = Statevector::from_instruction(qc_direct);
    auto sv_bridge = Statevector::from_instruction(qc_bridge);

    std::cout << "Direct CX(0, 2): " << sv_direct << std::endl;
    std::cout << "Bridge form:    " << sv_bridge << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make bridge_verify && ./bridge_verify

### 演習4: ブリッジの等式を使って最近接 GHZ を構築する

以下はファンアウトパターンで構築した3量子ビット GHZ です。

```cpp
QuantumCircuit qc(3, 0);
qc.h(0);
qc.cx(0, 1);
qc.cx(0, 2);   // <- この CX は 0-1-2 チェーン上では非局所です！
```

最後のゲート `CX(0, 2)` は、隣接する量子ビットだけが接続されているハードウェアでは直接実行できません。

**課題:** 同じ3量子ビット GHZ 状態を構築してください。ただし **ブリッジの等式** を使って `CX(0, 2)` を4つの最近接 CX ゲートに置き換えます。最終的な回路は `CX(0, 1)` と `CX(1, 2)` だけを使い、`CX(0, 2)` は決して使わないようにしてください。

終わったら、チェーン形式（`H(0); CX(0, 1); CX(1, 2)`）に比べて何個の CX ゲートを使ったかを見てください。その差が、ハードウェアがチェーンを望んでいるのにファンアウトの形にこだわることの本当のコストです。

In [ ]:
%%writefile "{LAB_ROOT}/bridge_exercise/bridge_exercise.cpp"
// 演習4: 最近接 CX ゲート（ブリッジの等式）だけを使って3量子ビット GHZ を構築する

#include "compat/quantumcircuit.hpp"
#include <fstream>
#include "compat/statevector.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit qc(3, 0);
    qc.h(0);
    qc.cx(0, 1);

    // TODO: ここにあなたのコードを追加してください: ブリッジの等式を使って非局所の CX(0, 2) を置き換える

    qc.draw();

    // Python の qc_grader クライアントで採点するために OpenQASM 3 を出力します。
    std::ofstream qasm_out("../bridge_exercise/circuit.qasm");
    qasm_out << qc.to_qasm3();
    qasm_out.close();

    auto sv = Statevector::from_instruction(qc);
    std::cout << "Your state: " << sv << std::endl;

    // 確認: |000> と |111> がそれぞれ確率0.5の有効な3量子ビット GHZ 状態のはず
    auto data = sv.data();
    double prob_0 = std::norm(data[0]);
    double prob_last = std::norm(data.back());

    if (std::abs(prob_0 - 0.5) < 0.01 && std::abs(prob_last - 0.5) < 0.01) {
        std::cout << "Valid 3-qubit GHZ state!" << std::endl;
    } else {
        std::cout << "Not a valid GHZ state. Check your CX gates." << std::endl;
    }

    auto ops = qc.count_ops();
    std::cout << "CX gate count: " << ops["cx"] << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make bridge_exercise && ./bridge_exercise

In [ ]:
# C++ バイナリが書き出した QASM3 を再読み込みし、再構成した回路を既存の grader に
# 提出することで解答を採点します。
import qiskit.qasm3
from pathlib import Path
_qasm_path = Path(LAB_ROOT) / 'bridge_exercise' / 'circuit.qasm'
_qasm = _qasm_path.read_text() if _qasm_path.is_file() else ''
if not _qasm.strip():
    raise RuntimeError(
        f"{_qasm_path} が空か存在しません -- 採点する前に、（コードを埋めた状態で）"
        "上のビルド/実行セルを実行してください。"
    )
qc = qiskit.qasm3.loads(_qasm)
grade_lab1_ex4(qc)

<details>
<summary><b>クリックして解答を表示</b></summary>

```cpp
QuantumCircuit qc(3, 0);
qc.h(0);
qc.cx(0, 1);
// CX(0, 2) を置き換えるブリッジの等式: 4つの最近接 CX ゲート
qc.cx(0, 1);
qc.cx(1, 2);
qc.cx(0, 1);
qc.cx(1, 2);
```

これは機能しますが、元の `CX(0, 1)` とブリッジ由来の最初の `CX(0, 1)` が隣り合っていること、そして `CX(0, 1) * CX(0, 1) = I`（どの CX も自分自身の逆）であることに注目してください。したがってそれらを打ち消せて、次が残ります。

```cpp
qc.h(0);
qc.cx(1, 2);
qc.cx(0, 1);
qc.cx(1, 2);
```

**CX 3個。** チェーン形式 `H(0); CX(0,1); CX(1,2)` が **2個** を使うのと比べてください。簡約化しても、最近接チップ上でファンアウトの形にこだわると、非局所のホップごとに CX が1個余分にかかります。より長いチェーンや距離の大きい CX では、このオーバーヘッドは急速に増大します。だからこそ、最初からトポロジーを意識した回路を設計すること（パート3の残り）が本当の勝ち筋なのです。
</details>

### 演習5: 直線上の効率的な5量子ビット GHZ

FakeTorino 上の物理量子ビット 0〜4 はチェーンを形成します: 0-1-2-3-4。この接続性を尊重し、深さを最小化する GHZ 回路を構築しましょう。

**課題:** 真ん中の量子ビットから始めて、抽象的な深さ **4** の5量子ビット GHZ 回路を構築してください。

**ヒント:** パート2の「真ん中から始める」トリックを思い出してください！

In [ ]:
%%writefile "{LAB_ROOT}/ghz_line_exercise/ghz_line_exercise.cpp"
// 演習5: 直線上の効率的な5量子ビット GHZ
// 真ん中の量子ビットから始めて深さ4の GHZ を構築する

#include "compat/quantumcircuit.hpp"
#include <fstream>
#include "compat/statevector.hpp"
#include "compat/target.hpp"
#include "compat/transpile.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(5, 0);

    // TODO: ここにあなたのコードを追加してください: 真ん中から始めて深さ4の GHZ を構築

    circ.draw();

    auto d = circ.depth();
    std::cout << "Abstract depth: " << d << std::endl;

    if (d == 4) {
        std::cout << "Correct! Depth 4 achieved." << std::endl;
    } else {
        std::cout << "Expected depth 4, got " << d << ". Try again!" << std::endl;
    }

    // 直線の部分グラフ 0-1-2-3-4 向けにトランスパイル
    // Python の qc_grader クライアントで採点するために OpenQASM 3 を出力します。
    std::ofstream qasm_out("../ghz_line_exercise/circuit.qasm");
    qasm_out << circ.to_qasm3();
    qasm_out.close();

    auto circ_meas = circ.measure_all();

    auto target = make_torino_target();
    auto pm = generate_preset_pass_manager(1, target);
    auto isa_qc = pm.run(circ_meas);

    std::cout << "Transpiled depth: " << isa_qc.depth() << std::endl;
    auto ops = isa_qc.count_ops();
    std::cout << "CZ gates: " << (ops.count("cz") ? ops["cz"] : 0) << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_line_exercise && ./ghz_line_exercise

In [ ]:
# C++ バイナリが書き出した QASM3 を再読み込みし、再構成した回路を既存の grader に
# 提出することで解答を採点します。
import qiskit.qasm3
from pathlib import Path
_qasm_path = Path(LAB_ROOT) / 'ghz_line_exercise' / 'circuit.qasm'
_qasm = _qasm_path.read_text() if _qasm_path.is_file() else ''
if not _qasm.strip():
    raise RuntimeError(
        f"{_qasm_path} が空か存在しません -- 採点する前に、（コードを埋めた状態で）"
        "上のビルド/実行セルを実行してください。"
    )
qc = qiskit.qasm3.loads(_qasm)
grade_lab1_ex5(qc)

### ハードウェアのトポロジー: ヘビーヘックス

FakeTorino は **ヘビーヘックス**（heavy-hex）接続性を持つ **Heron プロセッサー** をモデル化します。133個の量子ビットは特徴的なパターンで配置されています。

- 各行を横切って量子ビットの **水平チェーン** が走る
- **垂直のブリッジ量子ビット** がずれた位置で行同士を接続する
- ほとんどの量子ビットは **次数2**（隣接が2つ）
- **分岐（ジャンクション）量子ビット** は **次数3**（隣接が3つ）を持つ。これらが分岐点

このトポロジーは全結合では *ありません*。隣接しない量子ビット間の2量子ビットゲートには SWAP ゲートが必要で、深さが劇的に増加します。ヘビーヘックス向けに効率的な回路を構築するとは、接続性に *逆らう* のではなく、*沿って* 作業することを意味します。

FakeTorino のカップリングマップの一部を可視化して、その構造を見てみましょう。

In [ ]:
%%writefile "{LAB_ROOT}/topology_viz/topology_viz.cpp"
// 量子ビット 0-45 のヘビーヘックス接続性を可視化する
// (演習6の q25 の T 字ジャンクション -- その3番目の隣接 q35 は最初の2行より
//  先にある -- が次数3として表示されるよう、範囲を 0-45 に拡張しています)

#include "compat/quantumcircuit.hpp"
#include "compat/target.hpp"
#include <nlohmann/json.hpp>
#include <fstream>

using namespace Qiskit::compat;

int main() {
    int n = 46;
    auto adj = torino_adjacency(n);

    std::cout << "FakeTorino: Qubits 0-45 (heavy-hex topology)" << std::endl;
    std::cout << std::endl;

    // 接続性を表示し、次数3のジャンクションを特定する
    std::vector<int> junctions;
    for (int q = 0; q < n; q++) {
        if (!adj[q].empty()) {
            std::cout << "  Qubit " << q << " -> {";
            for (size_t i = 0; i < adj[q].size(); i++) {
                if (i > 0) std::cout << ", ";
                std::cout << adj[q][i];
            }
            std::cout << "}";
            if (adj[q].size() >= 3) {
                std::cout << "  ** degree-3 junction **";
                junctions.push_back(q);
            }
            std::cout << std::endl;
        }
    }

    std::cout << "\nDegree-3 junctions (qubits 0-45):";
    for (int q : junctions) std::cout << " " << q;
    std::cout << std::endl;

    // Python の可視化セル用にグラフデータを JSON として書き出す
    nlohmann::json graph;
    graph["nodes"] = nlohmann::json::array();
    graph["edges"] = nlohmann::json::array();
    graph["junctions"] = junctions;

    for (int q = 0; q < n; q++)
        if (!adj[q].empty())
            graph["nodes"].push_back(q);

    for (int q = 0; q < n; q++)
        for (int nb : adj[q])
            if (q < nb)
                graph["edges"].push_back({q, nb});

    std::ofstream out("../topology_viz/graph.json");
    out << graph.dump() << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make topology_viz && ./topology_viz

In [ ]:
import json, networkx as nx, matplotlib.pyplot as plt

with open(f'{LAB_ROOT}/topology_viz/graph.json') as f:
    data = json.load(f)

G = nx.Graph()
G.add_nodes_from(data['nodes'])
G.add_edges_from(data['edges'])
junctions = set(data['junctions'])

colors = ['#ff6b6b' if q in junctions else '#4ecdc4' for q in G.nodes()]

fig, ax = plt.subplots(figsize=(14, 5))
pos = nx.kamada_kawai_layout(G)
nx.draw(G, pos, with_labels=True, node_size=400, font_size=9,
        node_color=colors, edge_color='#888', width=2, ax=ax)
ax.set_title('FakeTorino: Qubits 0-45 (red = degree-3 junctions)')
plt.tight_layout()
plt.show()

### 演習6: ヘビーヘックス部分グラフ上の7量子ビット GHZ

FakeTorino のこの T 字型の部分グラフを考えます。

```
23 -- 24 -- 25 -- 26 -- 27
            |
           35
            |
           44
```

量子ビット25は **次数3のジャンクション** です。量子ビット24、26、35に接続しています。

**課題:** この部分グラフ上に、**可能な限り最小の深さ** で7量子ビット GHZ 回路を構築してください。

**ヒント:**
- どの量子ビットから始めるべきでしょうか？（分岐点がどこにあるかを考えましょう！）
- 覚えておいてください: 1つの量子ビットは1つの層につき1つの CX ゲートにしか参加できません
- 抽象的な量子ビットを次のように対応づけます: 0→q23、1→q24、2→q25、3→q26、4→q27、5→q35、6→q44

In [ ]:
%%writefile "{LAB_ROOT}/ghz_hex_exercise/ghz_hex_exercise.cpp"
// 演習6: ヘビーヘックスの T 字型部分グラフ上の7量子ビット GHZ
//
// 部分グラフの隣接関係（抽象的な量子ビット番号）:
// 0(q23) -- 1(q24) -- 2(q25) -- 3(q26) -- 4(q27)
//                      |
//                    5(q35)
//                      |
//                    6(q44)

#include "compat/quantumcircuit.hpp"
#include <fstream>
#include "compat/statevector.hpp"
#include "compat/target.hpp"
#include "compat/transpile.hpp"

using namespace Qiskit::compat;

int main() {
    QuantumCircuit circ(7, 0);

    // TODO: ここにあなたのコードを追加してください: 最適な量子ビット（ジャンクション）から始めて GHZ を構築

    circ.draw();

    auto d = circ.depth();
    std::cout << "Abstract depth: " << d << std::endl;

    // 有効な7量子ビット GHZ 状態かどうかを確認する
    // Python の qc_grader クライアントで採点するために OpenQASM 3 を出力します。
    std::ofstream qasm_out("../ghz_hex_exercise/circuit.qasm");
    qasm_out << circ.to_qasm3();
    qasm_out.close();

    auto sv = Statevector::from_instruction(circ);
    auto data = sv.data();
    double prob_0 = std::norm(data[0]);
    double prob_last = std::norm(data.back());

    if (std::abs(prob_0 - 0.5) < 0.01 && std::abs(prob_last - 0.5) < 0.01) {
        std::cout << "Valid 7-qubit GHZ state!" << std::endl;
    } else {
        std::cout << "Not a valid GHZ state. Check your circuit." << std::endl;
    }

    // 物理レイアウトでトランスパイル
    auto circ_meas = circ.measure_all();

    auto target = make_torino_target();
    auto pm = generate_preset_pass_manager(1, target);
    auto isa_qc = pm.run(circ_meas);

    std::cout << "Transpiled depth: " << isa_qc.depth() << std::endl;
    auto ops = isa_qc.count_ops();
    std::cout << "CZ gates: " << (ops.count("cz") ? ops["cz"] : 0) << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_hex_exercise && ./ghz_hex_exercise

In [ ]:
# C++ バイナリが書き出した QASM3 を再読み込みし、再構成した回路を既存の grader に
# 提出することで解答を採点します。
import qiskit.qasm3
from pathlib import Path
_qasm_path = Path(LAB_ROOT) / 'ghz_hex_exercise' / 'circuit.qasm'
_qasm = _qasm_path.read_text() if _qasm_path.is_file() else ''
if not _qasm.strip():
    raise RuntimeError(
        f"{_qasm_path} が空か存在しません -- 採点する前に、（コードを埋めた状態で）"
        "上のビルド/実行セルを実行してください。"
    )
qc = qiskit.qasm3.loads(_qasm)
grade_lab1_ex6(qc)

### 演習7: 最終チャレンジ: 64量子ビット GHZ

さて、すべてを組み合わせましょう。あなたの課題は、**64量子ビット**（FakeTorino の量子ビット 0〜63）向けに効率的な GHZ 状態を構築し、トランスパイルすることです。

**戦略:**
1. バックエンドのカップリングマップを使って量子ビット 0〜63 の接続性を調べる
2. よく選ばれた中心量子ビットから BFS（幅優先探索）で **全域木** を構築する
3. 木に沿って層ごとにもつれさせて GHZ 回路を構成する
4. トランスパイルして、素朴な線形 GHZ と深さを比較する

**役立つ情報:**
- 0〜63 の部分グラフで最適な中心量子ビットは **量子ビット25** です（他のどの量子ビットへの最大距離も最小化します）
- `backend.coupling_map.neighbors(q)` は量子ビット $q$ の隣接を返します

In [ ]:
%%writefile "{LAB_ROOT}/ghz_64_exercise/ghz_64_exercise.cpp"
// 演習7: ヘビーヘックストポロジー上で BFS を使う64量子ビット GHZ

#include <queue>
#include <map>
#include "compat/quantumcircuit.hpp"
#include <fstream>
#include "compat/target.hpp"
#include "compat/transpile.hpp"

using namespace Qiskit::compat;

int main() {
    auto adj = torino_adjacency(64);
    int n = 64;
    int center = 25;  // これを変えてみてください！

    // 次数3のジャンクションを表示
    std::cout << "Degree-3 junctions in qubits 0-63:" << std::endl;
    for (int q = 0; q < n; q++) {
        if (adj[q].size() >= 3) {
            std::cout << "  Qubit " << q << ": neighbors {";
            for (size_t i = 0; i < adj[q].size(); i++) {
                if (i > 0) std::cout << ", ";
                std::cout << adj[q][i];
            }
            std::cout << "}" << std::endl;
        }
    }

    // ステップ1: center から BFS で全域木を構築
    std::map<int, int> parent;
    std::map<int, int> bfs_depth;
    parent[center] = -1;
    bfs_depth[center] = 0;
    std::queue<int> queue;
    queue.push(center);

    while (!queue.empty()) {
        int node = queue.front();
        queue.pop();
        for (int nb : adj[node]) {
            if (parent.find(nb) == parent.end()) {
                parent[nb] = node;
                bfs_depth[nb] = bfs_depth[node] + 1;
                queue.push(nb);
            }
        }
    }

    int max_d = 0;
    for (auto& kv : bfs_depth)
        max_d = std::max(max_d, kv.second);

    std::cout << "\nBFS from qubit " << center
              << ": reached " << parent.size() << " qubits" << std::endl;
    std::cout << "Max BFS depth: " << max_d << std::endl;

    // ステップ2: BFS 木に沿って層ごとに GHZ 回路を構築
    QuantumCircuit circ(n, 0);
    circ.h(center);

    for (int d = 1; d <= max_d; d++) {
        for (int q = 0; q < n; q++) {
            if (bfs_depth.count(q) && bfs_depth[q] == d) {
                // TODO: ここにあなたのコードを追加してください: parent[q] から q への CX を追加
            }
        }
    }

    std::cout << "\nYour GHZ abstract depth: " << circ.depth() << std::endl;

    // トランスパイルして素朴なチェーンと比較
    // Python の qc_grader クライアントで採点するために OpenQASM 3 を出力します。
    std::ofstream qasm_out("../ghz_64_exercise/circuit.qasm");
    qasm_out << circ.to_qasm3();
    qasm_out.close();

    auto circ_meas = circ.measure_all();

    auto target = make_torino_target();
    auto pm = generate_preset_pass_manager(1, target);
    auto isa_qc = pm.run(circ_meas);

    auto ops = isa_qc.count_ops();
    std::cout << "Your GHZ transpiled depth: " << isa_qc.depth() << std::endl;
    std::cout << "Your GHZ CZ gates: " << (ops.count("cz") ? ops["cz"] : 0) << std::endl;

    // 比較用の素朴なチェーン
    QuantumCircuit naive(n, 0);
    naive.h(0);
    for (int q = 0; q < n - 1; q++)
        naive.cx(q, q + 1);
    auto naive_meas = naive.measure_all();

    auto isa_naive = pm.run(naive_meas);
    auto ops_naive = isa_naive.count_ops();
    std::cout << "\nNaive chain transpiled depth: " << isa_naive.depth() << std::endl;
    std::cout << "Naive chain CZ gates: " << (ops_naive.count("cz") ? ops_naive["cz"] : 0) << std::endl;

    return 0;
}

In [ ]:
!cd "{LAB_ROOT}/build" && make ghz_64_exercise && ./ghz_64_exercise

In [ ]:
# C++ バイナリが書き出した QASM3 を再読み込みし、再構成した回路を既存の grader に
# 提出することで解答を採点します。
import qiskit.qasm3
from pathlib import Path
_qasm_path = Path(LAB_ROOT) / 'ghz_64_exercise' / 'circuit.qasm'
_qasm = _qasm_path.read_text() if _qasm_path.is_file() else ''
if not _qasm.strip():
    raise RuntimeError(
        f"{_qasm_path} が空か存在しません -- 採点する前に、（コードを埋めた状態で）"
        "上のビルド/実行セルを実行してください。"
    )
qc = qiskit.qasm3.loads(_qasm)
grade_lab1_ex7(qc)

### パート3のまとめ

**達成した学習目標:**
- **トランスパイル** は抽象的な回路をハードウェアネイティブなものへ変換する（ゲート分解 + 量子ビットルーティング + レイアウト）
- Heron プロセッサーは $\{R_Z, \sqrt{X}, X, CZ\}$ を使う（H も CX もない）
- **ヘビーヘックストポロジー**: 次数2のチェーン量子ビットと次数3のジャンクション量子ビット
- SWAP ゲート（隣接しない量子ビットに必要）はそれぞれ 3 個の CZ のコストがかかるので、トポロジーを意識した回路を設計して避ける
- **BFS 全域木**: 任意のハードウェアグラフ上で効率的な GHZ 回路を構築する体系的な方法

### 先を見据えて: Lab 2

Lab 2 では、新しい IBM **Nighthawk** プロセッサーと、（このラボで扱ってきたアーキテクチャーである）**Heron** プロセッサーを比較します。この2つは異なる接続性パターンを持ちます。Heron はヘビーヘックスを使う一方、Nighthawk は量子ビットあたりの隣接がより多い、より密なグラフを持ちます。こうしたアーキテクチャーの違いは、回路の性能に直接影響します。

ここで身につけたスキル（効率的な GHZ 状態の構築、深さの理解、ヘビーヘックストポロジーでの作業）は、GHZ 回路を使って2つのプロセッサーをベンチマークし比較する Lab 2 に直接つながります。

Lab 2 でお会いしましょう！ 🚀

---

## Lab 1 完了: 最終まとめ

**学んだこと:**

| 概念 | 要点 |
|---------|-------------|
| **X, H, CX ゲート** | 量子回路の構成要素 |
| **重ね合わせ** | H ゲートが $\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ を生成 |
| **量子もつれ** | CX + 重ね合わせが相関した量子ビットのペアを生成 |
| **ベル状態と GHZ 状態** | 2量子ビットと N 量子ビットの量子もつれ |
| **回路の深さ** | 順次実行されるゲート層の数。少ないほど = ノイズが少ない |
| **再帰的ファンアウト** | GHZ を $O(N)$ ではなく $O(\log N)$ の深さで |
| **トランスパイル** | 抽象的 → ハードウェアネイティブな回路 |
| **ヘビーヘックストポロジー** | 次数2のチェーン + 次数3のジャンクション |
| **トポロジーを意識した設計** | ハードウェアの接続性に沿って SWAP を避ける |

# 追加情報

**作成者:** James Weaver

**バージョン:** 1.0.0